# Task : Mental Health Support Chatbot (Fine-Tuned)

## Objective
Build an empathetic chatbot for stress, anxiety, and emotional wellness by fine-tuning a small language model.

## Model Base
DistilGPT2 (lightweight, fast) — can be swapped for GPT-Neo-125M

## Dataset
EmpatheticDialogues by Facebook AI (via Hugging Face Datasets)

## Step 1: Install Libraries

**Note:** Fine-tuning requires a GPU for reasonable speed. Use **Google Colab** (free GPU) or Kaggle Notebooks.

In [ ]:
!pip install transformers datasets accelerate torch sentencepiece

## Step 2: Import Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import warnings
warnings.filterwarnings('ignore')

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('⚠️  No GPU detected. Training will be slow. Consider using Google Colab with GPU runtime.')

## Step 3: Load EmpatheticDialogues Dataset

In [ ]:
# Load Facebook AI's EmpatheticDialogues dataset
print('Loading EmpatheticDialogues dataset...')
dataset = load_dataset('empathetic_dialogues')

print('Dataset structure:')
print(dataset)
print(f'\nTrain samples : {len(dataset["train"])}')
print(f'Val samples   : {len(dataset["validation"])}')
print(f'Test samples  : {len(dataset["test"])}')
print(f'\nSample entry:')
print(dataset['train'][0])

In [ ]:
# Explore emotion distribution
train_df = dataset['train'].to_pandas()

plt.figure(figsize=(14, 5))
emotion_counts = train_df['context'].value_counts().head(20)
plt.bar(emotion_counts.index, emotion_counts.values, color='mediumpurple', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Emotion Categories in Training Data', fontsize=13, fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Total unique emotions: {train_df["context"].nunique()}')

## Step 4: Preprocess Dataset

In [ ]:
# Format: convert dialogue pairs into 'prompt -> response' text format
# We format as: 'Emotion: <emotion>\nPerson: <prompt>\nAssistant: <response>'

def format_conversation(example):
    """
    Format EmpatheticDialogues entry into a training text string.
    EmpatheticDialogues has 'prompt' and 'utterance' fields.
    """
    emotion  = example.get('context', 'neutral')
    prompt   = example.get('prompt', '')
    response = example.get('utterance', '')
    
    # Create training text
    text = (
        f'Emotion: {emotion}\n'
        f'Person: {prompt}\n'
        f'Assistant: {response}\n'
        f'<|endoftext|>'
    )
    return {'text': text}

# Apply formatting
print('Formatting dataset...')
formatted = dataset.map(format_conversation, remove_columns=dataset['train'].column_names)

print('\nSample formatted entry:')
print(formatted['train'][0]['text'])

## Step 5: Load Tokenizer and Model

In [ ]:
MODEL_NAME = 'distilgpt2'  # Lightweight — swap with 'EleutherAI/gpt-neo-125m' for better quality

print(f'Loading tokenizer and model: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 style models don't have a pad token — use eos_token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)

print(f'Model parameters: {model.num_parameters():,}')
print('Model and tokenizer loaded!')

In [ ]:
# Tokenize dataset
MAX_LENGTH = 128  # Keep short for faster training

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length'
    )

print('Tokenizing dataset (this may take a minute)...')

# Use a subset for faster training — increase for better results
tokenized = formatted.map(
    tokenize_function, batched=True, remove_columns=['text']
)

# Use a small subset if CPU-only
if device == 'cpu':
    print('CPU mode: using 500 train / 100 val samples for demo')
    train_dataset = tokenized['train'].select(range(500))
    eval_dataset  = tokenized['validation'].select(range(100))
else:
    print('GPU mode: using 5000 train / 500 val samples')
    train_dataset = tokenized['train'].select(range(5000))
    eval_dataset  = tokenized['validation'].select(range(500))

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

## Step 6: Fine-Tune the Model

In [ ]:
# Data collator for causal language modeling (no masking)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Training configuration
training_args = TrainingArguments(
    output_dir='./mental_health_chatbot',
    overwrite_output_dir=True,
    num_train_epochs=3,              # Increase for better quality
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=50,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),  # Use mixed precision on GPU
    report_to='none'
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print('Training configuration ready!')
print(f'Epochs: {training_args.num_train_epochs}')
print(f'Batch size: {training_args.per_device_train_batch_size}')

In [ ]:
# Start fine-tuning
print('Starting fine-tuning...\n')
train_result = trainer.train()

print('\nTraining complete!')
print(f'Training loss: {train_result.training_loss:.4f}')

In [ ]:
# Save the fine-tuned model
SAVE_PATH = './mental_health_chatbot_final'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

## Step 7: Generate Empathetic Responses

In [ ]:
def generate_empathetic_response(prompt, emotion='anxious', max_new_tokens=100):
    """
    Generate an empathetic response given a user message and detected emotion.
    """
    # Format input in the same structure used during training
    input_text = f'Emotion: {emotion}\nPerson: {prompt}\nAssistant:'
    
    inputs = tokenizer(input_text, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract only the assistant's response
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)
    
    if 'Assistant:' in full_output:
        response = full_output.split('Assistant:')[-1].strip()
        # Stop at next 'Person:' if present
        if 'Person:' in response:
            response = response.split('Person:')[0].strip()
        return response
    return full_output.strip()


# Test the fine-tuned model
test_prompts = [
    ('I feel so anxious about my exams and can\'t sleep.',    'anxious'),
    ('I\'ve been really stressed at work lately.',             'stressed'),
    ('I feel lonely and nobody understands me.',              'lonely'),
    ('I\'m proud of myself for completing a hard challenge!', 'proud'),
]

print('Testing Fine-Tuned Mental Health Chatbot\n')
print('='*55)
for prompt, emotion in test_prompts:
    print(f'\nEmotion : {emotion}')
    print(f'Person  : {prompt}')
    response = generate_empathetic_response(prompt, emotion)
    print(f'Bot     : {response}')

## Step 8: Streamlit App (Save and Run Separately)

In [ ]:
# Save this as app.py and run with: streamlit run app.py

streamlit_code = '''
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load fine-tuned model
@st.cache_resource
def load_model():
    path = './mental_health_chatbot_final'
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

tokenizer, model = load_model()

EMOTIONS = ['anxious', 'stressed', 'lonely', 'sad', 'angry',
            'overwhelmed', 'proud', 'grateful', 'neutral']

st.set_page_config(page_title='Mental Health Support Bot', page_icon='💬')
st.title('💬 Mental Health Support Chatbot')
st.markdown('A safe space to share how you are feeling. I am here to listen.')

# Sidebar
with st.sidebar:
    st.header('Settings')
    emotion = st.selectbox('How are you feeling?', EMOTIONS)
    st.markdown('---')
    st.info('This chatbot provides emotional support only. For serious mental health concerns, please consult a professional.')

# Session state for chat history
if 'messages' not in st.session_state:
    st.session_state.messages = []

# Display chat history
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.write(msg['content'])

# Chat input
if user_input := st.chat_input('Share what is on your mind...'):
    # Display user message
    st.session_state.messages.append({'role': 'user', 'content': user_input})
    with st.chat_message('user'):
        st.write(user_input)
    
    # Generate response
    input_text = f"Emotion: {emotion}\\nPerson: {user_input}\\nAssistant:"
    inputs = tokenizer(input_text, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=120, do_sample=True,
                                temperature=0.8, top_p=0.9,
                                pad_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(output[0], skip_special_tokens=True)
    response = full.split('Assistant:')[-1].strip().split('Person:')[0].strip()
    
    # Display bot response
    st.session_state.messages.append({'role': 'assistant', 'content': response})
    with st.chat_message('assistant'):
        st.write(response)
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print('Streamlit app saved as app.py')
print('To run: streamlit run app.py')

## Step 9: Conclusion

- **DistilGPT2** was fine-tuned on Facebook AI's **EmpatheticDialogues** dataset to generate supportive responses.
- Training data was formatted as `Emotion → Person → Assistant` to teach the model context-aware empathy.
- The model learns to produce **emotionally aware, gentle, and supportive replies**.
- A **Streamlit app** provides a user-friendly interface for testing.
- **Key takeaways**: Fine-tuning even a small model on domain-specific data significantly improves tone.
- **Next steps**: Use GPT-Neo-1.3B or Mistral-7B for higher quality, or add RLHF for better alignment.
- **Safety reminder**: This is a support chatbot, NOT a substitute for professional mental health care.